<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/Chapter2_Demo2_VectorDB_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Database Demo — Medical Textbooks

This notebook builds a semantic vector database over medical textbooks using:
- **sentence-transformers** (`all-MiniLM-L6-v2`) for embeddings (PyTorch, no TF conflict)
- **FAISS** for fast nearest-neighbour search
- **gensim + textblob** for text preprocessing

**Changes from original:**
- Replaced `TFAutoModel` (TensorFlow) with `SentenceTransformer` — eliminates ml-dtypes/jax conflicts
- Simplified embedding matrix construction (no redundant `.flatten()`)
- Added FAISS index persistence (save/load)
- Added chunk metadata tracking so retrieved results show source file + chunk index

## 1. Install Dependencies

In [1]:
# Core deps only — no tensorflow needed anymore
!pip install requests tqdm faiss-cpu sentence-transformers textblob gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.8 MB/s eta 0:00:00


## 2. Download Dataset

In [2]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path("./mimic_textbooks")

def download_and_extract_zip(url, extract_to=DATA_DIR):
    extract_to.mkdir(parents=True, exist_ok=True)
    zip_path = extract_to / "textbooks.zip"

    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(zip_path, "wb") as f, tqdm(
        total=total, unit='B', unit_scale=True, desc="textbooks.zip"
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Done.")

dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
download_and_extract_zip(dataset_url)

textbooks.zip: 100%|██████████| 90.2M/90.2M [00:04<00:00, 21.0MB/s]


Extracting dataset...
Done.


## 3. Load, Clean & Chunk Documents

In [3]:
import re
from gensim.utils import simple_preprocess

CHUNK_SIZE = 200  # words per chunk

def load_text_files(directory):
    """Load all .txt files from a directory, returning (filename, text) tuples."""
    files = []
    for file_path in sorted(Path(directory).glob("*.txt")):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            files.append((file_path.name, f.read()))
    return files

def clean_and_tokenize(text):
    """Basic cleaning: normalise whitespace, lowercase, remove special chars."""
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

def chunk_text(text, chunk_size=CHUNK_SIZE):
    """Split text into fixed-size word chunks."""
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, and chunk — also track metadata (source file + chunk index)
raw_files = load_text_files(DATA_DIR / "textbooks/en")
print(f"Loaded {len(raw_files)} text files.")

chunked_documents = []  # list of chunk strings
chunk_metadata = []     # list of {"source": filename, "chunk_index": i}

for filename, text in raw_files:
    cleaned = clean_and_tokenize(text)
    chunks = chunk_text(cleaned)
    for i, chunk in enumerate(chunks):
        chunked_documents.append(chunk)
        chunk_metadata.append({"source": filename, "chunk_index": i})

print(f"Total document chunks: {len(chunked_documents)}")

# NOTE: Spell correction (textblob) is intentionally skipped.
# It is extremely slow on large medical corpora and rarely helps
# semantic similarity models which handle minor noise well.

Loaded 18 text files.
Total document chunks: 60061


## 4. Generate Embeddings

Using `SentenceTransformer` directly — simpler API, pure PyTorch, no TensorFlow conflicts.

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model — downloads ~90MB on first run, cached afterwards
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Encode all chunks in batches with a progress bar
# Output shape: (num_chunks, 384)
embeddings = model.encode(
    chunked_documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2-normalise — cosine sim becomes dot product
)

print(f"Embeddings shape: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_653/548443532.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/470 [00:00<?, ?it/s]

Embeddings shape: (60061, 384)


## 5. Build FAISS Index

In [5]:
import faiss

dimension = embeddings.shape[1]  # 384 for MiniLM

# Because embeddings are L2-normalised above, IndexFlatIP (inner product)
# is equivalent to cosine similarity — faster and more meaningful than L2.
index = faiss.IndexFlatIP(dimension)

# Cast to float32 — FAISS requirement
embedding_matrix = embeddings.astype('float32')
index.add(embedding_matrix)

print(f"Total embeddings indexed: {index.ntotal}")

Total embeddings indexed: 60061


## 6. Save Index to Disk

So you don't have to re-embed on every session.

In [6]:
import pickle

INDEX_PATH = "vectordb.index"
META_PATH  = "vectordb_metadata.pkl"
DOCS_PATH  = "vectordb_chunks.pkl"

# Save FAISS index
faiss.write_index(index, INDEX_PATH)

# Save chunks and metadata alongside
with open(META_PATH, "wb") as f:
    pickle.dump(chunk_metadata, f)
with open(DOCS_PATH, "wb") as f:
    pickle.dump(chunked_documents, f)

print(f"Saved index  → {INDEX_PATH}")
print(f"Saved metadata → {META_PATH}")
print(f"Saved chunks   → {DOCS_PATH}")

Saved index  → vectordb.index
Saved metadata → vectordb_metadata.pkl
Saved chunks   → vectordb_chunks.pkl


## 7. Load Index from Disk (run this in future sessions instead of steps 2-6)

In [ ]:
# Uncomment to reload a saved index without re-embedding:

# import faiss, pickle
# from sentence_transformers import SentenceTransformer
#
# model          = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# index          = faiss.read_index("vectordb.index")
# chunk_metadata = pickle.load(open("vectordb_metadata.pkl", "rb"))
# chunked_documents = pickle.load(open("vectordb_chunks.pkl", "rb"))
# print(f"Loaded index with {index.ntotal} vectors.")

## 8. Query the Vector Database

In [7]:
def search(query_text, k=5):
    """
    Embed a query and retrieve the top-k most similar document chunks.
    Returns a list of dicts with keys: rank, score, source, chunk_index, text.
    """
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype('float32')

    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append({
            "rank":        rank,
            "score":       float(score),
            "source":      chunk_metadata[idx]["source"],
            "chunk_index": chunk_metadata[idx]["chunk_index"],
            "text":        chunked_documents[idx]
        })
    return results


# --- Example query ---
query = "What are causes of heart failure?"
results = search(query, k=5)

print(f"Query: {query}\n{'='*60}")
for r in results:
    print(f"\nRank {r['rank']}  |  Score: {r['score']:.4f}  |  {r['source']} (chunk {r['chunk_index']})")
    print(r['text'][:400], "..." if len(r['text']) > 400 else "")

Query: What are causes of heart failure?

Rank 1  |  Score: 0.6438  |  Pathology_Robbins.txt (chunk 1036)
down to six principal mechanisms failure of the pump in the most common situation the cardiac muscle contracts weakly and the chambers cannot empty systolic dysfunction in some cases the muscle cannot relax sufficiently to permit ventricular filling resulting in diastolic dysfunction obstruction to flow lesions that prevent valve opening eg calcific aortic valve stenosis or cause increased ventric ...

Rank 2  |  Score: 0.6163  |  Pathology_Robbins.txt (chunk 1037)
disease and typically is progressive condition with poor prognosis in the united states alone over million individuals are affected resulting in well over million annually and financial burden in excess of billion roughly one half of patients die within years of receiving diagnosis of chf and in deaths in the united states include heart failure as contributory cause chf occurs when the heart canno ...

Rank 3  |  Score: 